# SHD — Convolutional vs Fully-Connected SNN Comparison

Compare convolutional (Conv2D+LIF) and fully-connected (FC+LIF) SNN
architectures on the SHD spoken digits dataset.  The convolutional model
uses the full 700-channel input; the FC model downsamples to 128 channels.

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"

import jax
import jax.numpy as jnp
import haiku as hk
import optax
import spyx
import spyx.nn as snn
import jmp
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
%matplotlib notebook

In [ ]:
policy = jmp.get_policy('half')
hk.mixed_precision.set_policy(hk.Linear,  policy)
hk.mixed_precision.set_policy(hk.Conv2D, policy)
hk.mixed_precision.set_policy(snn.LIF,   policy)
hk.mixed_precision.set_policy(snn.LI,    policy)

### SHD Dataloading

Load SHD at two resolutions: 700-channel FC (downsampled) and 700-channel
raw for the convolutional model.

In [ ]:
import tonic
from tonic import datasets, transforms
from torch.utils.data import DataLoader

n_classes  = 20
sample_T   = 256
shd_dt     = 1e-6   # 1 µs per timestep
net_dt     = 1 / sample_T

In [ ]:
class _SHD2Raster:
    """Rasterise SHD events into packed-bit frames."""
    def __init__(self, encoding_dim, sample_T=100):
        self.encoding_dim = encoding_dim
        self.sample_T     = sample_T

    def __call__(self, events):
        tensor = np.zeros((events["t"].max() + 1, self.encoding_dim), dtype=int)
        np.add.at(tensor, (events["t"], events["x"]), 1)
        tensor = tensor[:self.sample_T, :]
        tensor = np.minimum(tensor, 1)
        return np.packbits(tensor, axis=0)


# FC model: downsample to 128 channels
fc_channels = 128
fc_transform = transforms.Compose([
    transforms.Downsample(time_factor=shd_dt/net_dt, spatial_factor=fc_channels/700),
    _SHD2Raster(fc_channels, sample_T=sample_T)
])

# Conv model: keep full 700 channels reshaped as (700, 1) spatial
conv_transform = transforms.Compose([
    transforms.Downsample(time_factor=shd_dt/net_dt),
    _SHD2Raster(700, sample_T=sample_T)
])

fc_train = tonic.datasets.SHD('./data', train=True, transform=fc_transform)
fc_test  = tonic.datasets.SHD('./data', train=False, transform=fc_transform)

conv_train = tonic.datasets.SHD('./data', train=True, transform=conv_transform)
conv_test  = tonic.datasets.SHD('./data', train=False, transform=conv_transform)

def make_loader(dataset, batch_size):
    dl = iter(DataLoader(dataset, batch_size=len(dataset),
                        collate_fn=tonic.collation.PadTensors(batch_first=True),
                        drop_last=True, shuffle=False))
    return next(dl)

x_fc_train, y_fc_train = make_loader(fc_train, len(fc_train))
x_fc_test,  y_fc_test  = make_loader(fc_test,  len(fc_test))

x_conv_train, y_conv_train = make_loader(conv_train, len(conv_train))
x_conv_test,  y_conv_test  = make_loader(conv_test,  len(conv_test))

x_fc_train  = jnp.array(x_fc_train,  dtype=jnp.uint8)
y_fc_train  = jnp.array(y_fc_train,  dtype=jnp.uint8)
x_fc_test   = jnp.array(x_fc_test,   dtype=jnp.uint8)
y_fc_test   = jnp.array(y_fc_test,   dtype=jnp.uint8)

x_conv_train = jnp.array(x_conv_train, dtype=jnp.uint8)
y_conv_train = jnp.array(y_conv_train, dtype=jnp.uint8)
x_conv_test  = jnp.array(x_conv_test,  dtype=jnp.uint8)
y_conv_test  = jnp.array(y_conv_test,  dtype=jnp.uint8)

print(f"FC  train: {x_fc_train.shape}   test: {x_fc_test.shape}")
print(f"Conv train: {x_conv_train.shape}  test: {x_conv_test.shape}")

### Models

In [ ]:
surrogate = spyx.axn.Axon(spyx.axn.arctan())

# ── Fully-Connected SNN ──────────────────────────────────────────────────
def snn_fc(x):
    """FC SNN: Linear→LIF→Linear→LIF→Linear→LI."""
    x = hk.BatchApply(hk.Linear(256, with_bias=False))(x)
    core = hk.DeepRNN([
        snn.LIF((256,), activation=surrogate),
        hk.Linear(256, with_bias=False),
        snn.LIF((256,), activation=surrogate),
        hk.Linear(n_classes, with_bias=False),
        snn.LI((n_classes,))
    ])
    spikes, V = hk.dynamic_unroll(
        core, x, core.initial_state(x.shape[0]),
        time_major=False, unroll=16
    )
    return spikes, V


# ── Convolutional SNN ────────────────────────────────────────────────────
def snn_conv(x):
    """
    Conv SNN: reshape (B,T,700) → (B,T,700,1) then Conv1D over time,
    followed by FC readout.
    """
    B, T, C = x.shape
    x = jnp.unpackbits(x, axis=1)           # (B, T, 700) → (B, T, 700) float
    x = x.astype(jnp.float32)
    # Treat time as spatial dim for Conv1D
    x = jnp.expand_dims(x, -1)               # (B, T, 700, 1)

    # Conv over time: 700 channels → 128 channels
    x = hk.Conv1D(128, kernel_shape=3, stride=1, padding='SAME')(x)
    x, _ = snn.LIF((128,), activation=surrogate)(x)

    # Global temporal average
    x = jnp.mean(x, axis=1)                  # (B, 128)

    # FC readout
    x = hk.Linear(256)(x)
    x, _ = snn.LIF((256,), activation=surrogate)(x)
    x = hk.Linear(n_classes)(x)
    spikes, V = snn.LI((n_classes,))(x)
    return spikes, V

In [ ]:
SNN_FC   = hk.without_apply_rng(hk.transform(snn_fc))
SNN_Conv = hk.without_apply_rng(hk.transform(snn_conv))

key = jax.random.PRNGKey(0)

params_fc   = SNN_FC.init(rng=key, x=x_fc_train[:128])
params_conv = SNN_Conv.init(rng=key, x=x_conv_train[:128])

n_params_fc   = sum(p.size for p in jax.tree_util.tree_leaves(params_fc))
n_params_conv = sum(p.size for p in jax.tree_util.tree_leaves(params_conv))
print(f"FC params:   {n_params_fc:,}")
print(f"Conv params: {n_params_conv:,}")

### Training loop helper

In [ ]:
def make_train_fn(SNN, opt):
    @jax.jit
    def net_eval(weights, events, targets):
        traces, _ = SNN.apply(weights, events)
        return spyx.fn.integral_crossentropy(traces, targets, smoothing=0.2)

    surrogate_grad = jax.value_and_grad(net_eval)

    @jax.jit
    def train_step(state, batch):
        weights, opt_state = state
        events, targets = batch
        events = jnp.unpackbits(events, axis=1)
        loss, grads = surrogate_grad(weights, events, targets)
        updates, opt_state = opt.update(grads, opt_state, weights)
        weights = optax.apply_updates(weights, updates)
        return (weights, opt_state), loss

    @jax.jit
    def eval_step(weights, batch):
        events, targets = batch
        events = jnp.unpackbits(events, axis=1)
        traces, _ = SNN.apply(weights, events)
        acc, _ = spyx.fn.integral_accuracy(traces, targets)
        loss   = net_eval(weights, events, targets)
        return jnp.array([acc, loss])

    return train_step, eval_step


def train_model(SNN, params, x_train, y_train, x_test, y_test,
                 label, epochs=120, batch=256, lr=2e-4):
    opt = optax.chain(optax.centralize(), optax.lion(learning_rate=lr))
    opt_state = opt.init(params)
    train_step, eval_step = make_train_fn(SNN, opt)

    n_train = y_train.shape[0]
    indices = jnp.arange(n_train)
    best_acc   = 0.0
    best_weights = None
    history = []
    rng = hk.next_rng_key()()

    for ep in tqdm(range(epochs), desc=label):
        rng, = jax.random.split(rng)
        perm = jax.random.permutation(rng, indices)
        epoch_losses = []

        for i in range(0, n_train - batch + 1, batch):
            batch_idx = perm[i:i+batch]
            (params, opt_state), loss = train_step(
                (params, opt_state),
                (x_train[batch_idx], y_train[batch_idx])
            )
            epoch_losses.append(loss)

        acc, val_loss = eval_step(params, (x_test, y_test))
        acc  = float(acc)
        loss = float(jnp.mean(jnp.array(epoch_losses)))
        history.append(dict(epoch=ep, loss=loss, acc=acc))

        if acc > best_acc:
            best_acc    = acc
            best_weights = params

        if ep % 30 == 0:
            print(f"  Epoch {ep:3d} | loss {loss:.4f} | acc {acc:.2%} | best {best_acc:.2%}")

    print(f"  → {label} best: {best_acc:.2%}")
    return best_weights, best_acc, history

In [ ]:
print("Training FC model...")
best_fc, acc_fc, hist_fc = train_model(
    SNN_FC, params_fc,
    x_fc_train, y_fc_train, x_fc_test, y_fc_test,
    "FC SNN"
)

print("\nTraining Conv model...")
best_conv, acc_conv, hist_conv = train_model(
    SNN_Conv, params_conv,
    x_conv_train, y_conv_train, x_conv_test, y_conv_test,
    "Conv SNN"
)

### Results comparison

In [ ]:
print(f"\n{'='*40}")
print(f"  FC SNN   best accuracy: {acc_fc:.2%}   ({n_params_fc:,} params)")
print(f"  Conv SNN best accuracy: {acc_conv:.2%}  ({n_params_conv:,} params)")
print(f"{'='*40}")

# Plot training curves side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, hist, acc in [('FC SNN', hist_fc, acc_fc),
                           ('Conv SNN', hist_conv, acc_conv)]:
    epochs = [h['epoch'] for h in hist]
    losses = [h['loss'] for h in hist]
    vaccs  = [h['acc']  for h in hist]
    ax = axes[0] if label == 'FC SNN' else axes[1]
    ax.plot(epochs, vaccs)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Test accuracy')
    ax.set_title(f'{label} (best={acc:.2%})'); ax.grid(True)

plt.suptitle('SHD — FC vs Conv SNN')
plt.tight_layout()
plt.show()

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, SNN_m, params_m, x_t, y_t, label in [
    (axes[0], SNN_FC,   best_fc,   x_fc_test,   y_fc_test,   'FC SNN'),
    (axes[1], SNN_Conv, best_conv, x_conv_test,  y_conv_test,  'Conv SNN'),
]:
    x_u = jnp.unpackbits(x_t, axis=1)
    spikes, _ = SNN_m.apply(params_m, x_u)
    preds = spyx.fn.integral_readout(spikes)
    cm = confusion_matrix(y_t, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=range(n_classes))
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(f'{label} confusion matrix')

plt.suptitle('SHD — FC vs Conv SNN confusion matrices')
plt.tight_layout()
plt.show()